# TP2 — SOLUTION (instructor version)
## Observations already available on Kabre: maps, time series, vertical section + intro to NCO/CDO

**This is the completed answer key**, matching `TP02_observations_NCO_CDO_EN.ipynb` (the student version with blanks). Keep this for yourself — don't hand it out before the session, or students will just copy-paste instead of working through the exercises.

> ⚠️ **Note for the instructor**: the paths and file names below are **placeholders** to be adapted to where the data actually lives on Kabre. Run this notebook once yourself beforehand to confirm the exercises actually work with the real files/variable names.

## 0. Where is the data?

In [ ]:
# TODO (instructor): adapt this path to the actual location on Kabre
OBS_DIR = "/data/gcambon/COMMONDATA/obs"
OBS_DIR = '/data0/user/gcambon/DATA/DATASETS_PSF/'

!ls -lh $OBS_DIR

## 1. Quick anatomy check (applying Course 2)

This directly applies what you just saw in Course 2 (netCDF file structure): let's open a real file and locate its dimensions, variables, and attributes.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# TODO (instructor): actual name of the satellite SST file already present on Kabre
SST_FILE = f"{OBS_DIR}/CAMERICA_DS/sst_central_america_2023.nc"

ds_sst = xr.open_dataset(SST_FILE)
ds_sst

---
# Part A — Python / xarray

## A.1 2D horizontal map (mean SST)

In [ ]:
VAR_SST = "analysed_sst"   # adapt if needed

# Time mean
sst_mean = ds_sst[VAR_SST].mean(dim="time")

# Kelvin -> Celsius if needed
if float(sst_mean.max()) > 100:
    sst_mean = sst_mean - 273.15

fig, ax = plt.subplots(figsize=(9, 6), subplot_kw={"projection": ccrs.PlateCarree()})
sst_mean.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="turbo",
              cbar_kwargs={"label": "Mean SST (°C)"})
ax.add_feature(cfeature.LAND, facecolor="0.8", zorder=2)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.set_title("Mean SST — Central America region")
plt.show()

**Expected result**: values roughly between 24°C and 30°C over the Central America/Caribbean region.

## A.2 Time series at a point

In [ ]:
lon0, lat0 = -85.0, 9.5   # off the coast of Costa Rica

ts = ds_sst[VAR_SST].sel(longitude=lon0, latitude=lat0, method="nearest")
if float(ts.max()) > 100:
    ts = ts - 273.15

fig, ax = plt.subplots(figsize=(12, 4))
ts.plot(ax=ax)
ax.set_ylabel("SST (°C)")
ax.set_title(f"SST time series at ({lon0}, {lat0})")
plt.show()

**Expected result**: a clear seasonal cycle, warmer roughly April-May, cooler December-February.

## A.3 Vertical section (3D climatology)

In [ ]:
# TODO (instructor): actual name of the 3D climatology file already present on Kabre
WOA_FILE = f"{OBS_DIR}/GLOBAL_DS/WOA2009/temp_seas.cdf"

ds_woa = xr.open_dataset(WOA_FILE)
VAR_TEMP = "temperature"   # typical WOA name, check with ds_woa.data_vars

lat_section = 10.0

temp_section = ds_woa[VAR_TEMP].mean(dim="T", skipna=True).sel(Y=lat_section, method="nearest")
temp_section

In [ ]:

# fig, ax = plt.subplots(figsize=(10, 5))
# cf = temp_section.plot.contourf(x="X", y="Z", ax=ax, cmap="turbo",
#                                  cbar_kwargs={"label": "Temperature (°C)"})
# ax.invert_yaxis()   # surface at the top
# ax.set_title(f"Vertical temperature section — lat = {lat_section}°N")
# plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
cf = temp_section.plot.pcolormesh(
    x="X", y="Z", ax=ax, cmap="turbo",
    cbar_kwargs={"label": "Temperature (°C)"}
)
ax.invert_yaxis()
ax.set_xlim(-94, -84)   # ajustez selon la convention de longitude trouvée
ax.set_title(f"Vertical temperature section — lat = {lat_section}°N")
plt.show()

**Expected result**: thermocline visible as a tight color transition around 50-150m depth.

### 🚀 Advanced track A — solution

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

# for ax, lat_s in zip(axes, [5.0, 25.0]):
#     section = ds_woa[VAR_TEMP].mean(dim="T").sel(Y=lat_s, method="nearest")
#     section.plot.contourf(x="X", y="Z", ax=ax, cmap="turbo", add_colorbar=(ax is axes[-1]))
#     ax.invert_yaxis()
#     ax.set_title(f"lat = {lat_s}°N")
#     ax.set_xlim(-120, -84)   # ajustez selon la convention de longitude trouvée

# plt.tight_layout()
# plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cf = None
for ax, lat_s in zip(axes, [5.0, 25.0]):
    section = ds_woa[VAR_TEMP].mean(dim="T").sel(Y=lat_s, method="nearest")
    cf = section.plot.contourf(
        x="X", y="Z", ax=ax, cmap="turbo",
        add_colorbar=False   # on désactive la colorbar automatique partout
    )
    ax.invert_yaxis()
    ax.set_title(f"lat = {lat_s}°N")
    ax.set_xlim(-120, -105)

# Une seule colorbar horizontale sous les deux subplots
fig.colorbar(
    cf, ax=axes, orientation="horizontal",
    fraction=0.05, pad=0.12, label="Temperature (°C)"
)

plt.show()

# Discussion: the thermocline is typically shallower near the coast (5°N, closer to
# the Costa Rica/Panama upwelling zones) than further offshore (15°N), because coastal
# upwelling brings colder, deeper water closer to the surface.

---
# Part B — Command line: NCO and CDO

## B.1 NCO — `ncdump`, `ncks`, `ncra`

In [ ]:
#%%bash
!ncdump -h {SST_FILE}

In [ ]:
#%%bash
!ncks -O -d longitude,-90.0,-80.0 -d latitude,5.0,15.0 {SST_FILE} sst_costarica.nc
!ncdump -h sst_costarica.nc | head -20

In [ ]:
#%%bash
!ncks -O --mk_rec_dmn time sst_costarica.nc sst_costarica_rec.nc
!ncra -O sst_costarica_rec.nc sst_costarica_mean_nco.nc
!ncdump -v analysed_sst sst_costarica_mean_nco.nc | tail -20

In [ ]:
!ncwa -O -a time sst_costarica.nc sst_costarica_mean_nwa.nc

## B.2 CDO — `sinfo`, `timmean`, `sellonlatbox`

In [ ]:
#%%bash
!cdo sinfo {SST_FILE}

In [ ]:
#%%bash
!cdo sellonlatbox,-90.0,-80.0,5.0,15.0 {SST_FILE} sst_costarica_cdo.nc
!cdo timmean sst_costarica_cdo.nc sst_costarica_mean_cdo.nc
!cdo sinfo sst_costarica_mean_cdo.nc

### 🚀 Advanced track B — solution

In [ ]:
#%%bash
# Chained: anomaly = file - monthly climatology of file, in one command
!cdo -O ymonsub ${SST_FILE} -ymonmean ${SST_FILE} sst_anomaly.nc
!cdo sinfo sst_anomaly.nc

## B.3 Cross-check (critical thinking!)

In [ ]:
ds_check_cdo = xr.open_dataset("sst_costarica_mean_cdo.nc")
val_cdo = float(ds_check_cdo[VAR_SST].mean())
if val_cdo > 100:
    val_cdo -= 273.15

ds_check_nco = xr.open_dataset("sst_costarica_mean_nco.nc")
#ds_check_nco = xr.open_dataset("sst_costarica_mean_nwa.nc")
val_nco = float(ds_check_nco[VAR_SST].mean())
if val_nco > 100:
    val_nco -= 273.15

sst_costarica_xr = ds_sst[VAR_SST].sel(longitude=slice(-90, -80), latitude=slice(5, 15))
val_xr = float(sst_costarica_xr.mean())
if val_xr > 100:
    val_xr -= 273.15

for name, v in [("Python/xarray", val_xr), ("NCO", val_nco), ("CDO", val_cdo)]:
    print(f"{name:15s} : {v:.3f}")

# Note: if latitude is stored decreasing (90 -> -90) rather than increasing,
# slice(5, 15) silently returns an empty selection. Check ds_sst.latitude.values
# ordering if val_xr comes out as NaN, and use slice(15, 5) instead in that case.

**Expected result**: the 3 values should match to within ~0.01°C.

---
## D. Debugging exercise — solution

`ds_sst.data_vars` reveals the actual variable name is `analysed_sst`, not `sea_surface_temperature`.

In [ ]:
ds_sst.data_vars

In [ ]:
# Corrected line
variable_temperature = ds_sst["analysed_sst"].mean("time")
print(variable_temperature)

---
## E. Pair mini-synthesis — notes for the instructor

There's no single "correct" output here since students pick their own variable/region. When circulating, check that each pair has:
- A 2D map with a sensible colorbar range for the chosen variable.
- A time series that shows a plausible seasonal or spatial pattern (not flat/constant, not absurd units).
- Two independently computed means that agree — if they don't, that's the debugging opportunity to use, not a failure to fix for them immediately.